In [2]:
from datetime import datetime
import pandas as pd
import json

from preprocessing import year

In [3]:
df = pd.read_csv('./irish-property-sales.csv', encoding="latin1")

column_names = [
    'date_of_sale',
    'address',
    'county',
    'eircode',
    'price',
    'not_full_market_price',
    'vat_exclusive',
    'property_description',
    'propterty_size_description'
]


def get_county_province_mapping(county: str):
    county = county.lower().strip()
    county_to_province = {
        "galway": "Connacht",
        "leitrim": "Connacht",
        "mayo": "Connacht",
        "roscommon": "Connacht",
        "sligo": "Connacht",
        "carlow": "Leinster",
        "dublin": "Leinster",
        "kildare": "Leinster",
        "kilkenny": "Leinster",
        "laois": "Leinster",
        "longford": "Leinster",
        "louth": "Leinster",
        "meath": "Leinster",
        "offaly": "Leinster",
        "westmeath": "Leinster",
        "wexford": "Leinster",
        "wicklow": "Leinster",
        "clare": "Munster",
        "cork": "Munster",
        "kerry": "Munster",
        "limerick": "Munster",
        "tipperary": "Munster",
        "waterford": "Munster",
        "antrim": "Ulster",
        "armagh": "Ulster",
        "cavan": "Ulster",
        "donegal": "Ulster",
        "down": "Ulster",
        "fermanagh": "Ulster",
        "londonderry": "Ulster",
        "monaghan": "Ulster",
        "tyrone": "Ulster",
    }

    return county_to_province[county]


def clean_price_column(price: str):
    cleaned = price[2:-1].replace(',', '')
    return float(cleaned)


def clean_property_description(desc: str):
    if desc == 'Second-Hand Dwelling house /Apartment':
        return 'second-hand'
    return 'new'


def map_property_size(property_size_desc: str):
    if property_size_desc in ["less than 38 sq metres", "n?os l? n? 38 m?adar cearnach"]:
        return "small"
    if property_size_desc in ["greater than or equal to 38 sq metres and less than 125 sq metres",
                              "níos mó ná nó cothrom le 38 méadar cearnach agus níos lú ná 125 méadar cearnach"]:
        return "medium"
    if property_size_desc in ["greater than 125 sq metres", "greater than or equal to 125 sq metres"]:
        return "large"

    return ""


def convert_to_iso_date(date: str):
    for fmt in ("%d/%m/%y", "%d/%m/%Y"):
        try:
            return datetime.strptime(date, fmt)
        except ValueError:
            continue
    raise ValueError(f'Unknown date format: {date}')


def map_address(full_address: str):
    return full_address.replace(' ', '_').strip().lower()


def map_county(county: str):
    return 'Laoighis' if county == 'Laois' else county


df.columns = column_names

df['date_of_sale'] = df['date_of_sale'].map(convert_to_iso_date)
df['year'] = df['date_of_sale'].map(lambda date: date.year)
df['address'] = df['address'].map(map_address)
df['price'] = df['price'].map(clean_price_column)
df['province'] = df['county'].map(get_county_province_mapping)
df['county'] = df['county'].map(map_county)
df['not_full_market_price'] = df['not_full_market_price'].map(lambda s: s.lower())
df['vat_exclusive'] = df['vat_exclusive'].map(lambda s: True if s == 'Yes' else False)
df['property_size'] = df['propterty_size_description'].map(map_property_size)
df['property_description'] = df['property_description'].map(clean_property_description)

/var/folders/nh/yh9mg2mx39ng5t5l9bd8r2400000gn/T/ipykernel_57853/4226063268.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./irish-property-sales.csv', encoding="latin1")


In [4]:
df.head()

,date_of_sale,address,county,eircode,price,not_full_market_price,vat_exclusive,property_description,propterty_size_description,year,province,property_size
0,2010-01-01,"5_braemor_drive,_churchtown,_co.dublin",Dublin,NaN,343000.0,no,False,second-hand,NaN,2010,Leinster,
1,2010-01-03,"134_ashewood_walk,_summerhill_lane,_portlaoise",Laoighis,NaN,185000.0,no,True,new,greater than or equal to 38 sq metres and less...,2010,Leinster,medium
2,2010-01-04,"1_meadow_avenue,_dundrum,_dublin_14",Dublin,NaN,438500.0,no,False,second-hand,NaN,2010,Leinster,
3,2010-01-04,"1_the_haven,_mornington",Meath,NaN,400000.0,no,False,second-hand,NaN,2010,Leinster,
4,2010-01-04,"11_melville_heights,_kilkenny",Kilkenny,NaN,160000.0,no,False,second-hand,NaN,2010,Leinster,


In [5]:
mean_price_per_county_per_year = df.groupby(["county", "year"])["price"].mean().reset_index()
result = []

for county, sub in mean_price_per_county_per_year.groupby("county"):
    avg_list = [
        {
            "year": int(row["year"]),
            "average": float(row["price"]),
        }
        for _, row in sub.iterrows()
    ]

    result.append({
        "county": str(county),
        "averagePrice": avg_list,
    })

with open("meanPriceCounties.json", "w") as f:
    json.dump(result, f, indent=2)


In [6]:
grouped = (
    df.groupby(['year', 'property_description'])
    .size()
    .reset_index(name='count')
    .pivot(index='year', columns='property_description', values='count')
    .reset_index()
)
grouped.head()

property_description,year,new,second-hand
0,2010,5315,15686
1,2011,2956,15484
2,2012,3179,22193
3,2013,3896,26322
4,2014,5424,38246


In [7]:
result = []
for _, row in grouped.iterrows():
    result.append({
        'year': int(row['year']),
        'new': int(row['new']),
        'secondHand': int(row['second-hand']),

    })

print(result)
with open("numberOfPropertyTypesPerYear.json", "w") as f:
    json.dump(result, f, indent=2)


[{'year': 2010, 'new': 5315, 'secondHand': 15686}, {'year': 2011, 'new': 2956, 'secondHand': 15484}, {'year': 2012, 'new': 3179, 'secondHand': 22193}, {'year': 2013, 'new': 3896, 'secondHand': 26322}, {'year': 2014, 'new': 5424, 'secondHand': 38246}, {'year': 2015, 'new': 6245, 'secondHand': 42918}, {'year': 2016, 'new': 6879, 'secondHand': 43055}, {'year': 2017, 'new': 9359, 'secondHand': 45659}, {'year': 2018, 'new': 11103, 'secondHand': 46347}, {'year': 2019, 'new': 11360, 'secondHand': 47697}, {'year': 2020, 'new': 9447, 'secondHand': 40101}, {'year': 2021, 'new': 9571, 'secondHand': 50012}, {'year': 2022, 'new': 11003, 'secondHand': 51723}, {'year': 2023, 'new': 12111, 'secondHand': 51211}, {'year': 2024, 'new': 12936, 'secondHand': 48519}, {'year': 2025, 'new': 9978, 'secondHand': 36811}]


In [30]:
df_year_price = df[['year', 'price']]
result = []
for year in df_year_price['year'].unique():
    bins = [0, 100000, 200000, 300000, 400000, 500000, 750000, 1000000, 2000000]
    labels = ['0-100k', '100-200k', '200-300k', '300-400k', '400-500k', '500-750k', '750k-1M', '1M+']
    price_bins = pd.cut(df_year_price[df_year_price['year'] == year]['price'], bins=bins, labels=labels, include_lowest=True)
    print(price_bins)

    result.append({
        'year': int(year),
        'bins': bins,
        'priceBins': price_bins.value_counts().sort_index().to_list()
    })

print(result)
with open("priceBinsPerYear.json", "w") as f:
    json.dump(result, f, indent=2)

0        300-400k
1        100-200k
2        400-500k
3        300-400k
4        100-200k
           ...   
20996    300-400k
20997    200-300k
20998      0-100k
20999      0-100k
21000    100-200k
Name: price, Length: 21001, dtype: category
Categories (8, object): ['0-100k' < '100-200k' < '200-300k' < '300-400k' < '400-500k' < '500-750k' < '750k-1M' < '1M+']
21001    100-200k
21002    200-300k
21003    100-200k
21004    300-400k
21005    100-200k
           ...   
39436      0-100k
39437      0-100k
39438      0-100k
39439    200-300k
39440      0-100k
Name: price, Length: 18440, dtype: category
Categories (8, object): ['0-100k' < '100-200k' < '200-300k' < '300-400k' < '400-500k' < '500-750k' < '750k-1M' < '1M+']
39441    100-200k
39442    100-200k
39443    100-200k
39444      0-100k
39445    100-200k
           ...   
64808    100-200k
64809    200-300k
64810      0-100k
64811      0-100k
64812    100-200k
Name: price, Length: 25372, dtype: category
Categories (8, object): ['0-100k' 